# Week 2, day 5 (morning) — Worksheet 12 SOLUTIONS: lambda, map and filter   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Q10 is the one that costs people an afternoon. It prints a correct answer and
then an empty one, from the same object, with no error in between.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 12 — Lambda, map and filter. Run this once.
numbers = [1, 22, 35, 4, 5, 7, 8]
words = ["extract", "Load", "transform", "QA"]

staff = [
    {"name": "ana", "role": "engineer", "years": 6},
    {"name": "bo", "role": "analyst", "years": 2},
    {"name": "cai", "role": "engineer", "years": 9},
    {"name": "dee", "role": "analyst", "years": 4},
]

print("numbers:", numbers)
print("words:  ", words)

PART A — What a lambda is

### Question 1

`def` and `lambda`, side by side. -> `10`, `10`, `<class 'function'>`, `<function <lambda> at 0x…>`.

The same value, and the same **type**: a lambda produces an ordinary
function object, not some lesser thing. The only real differences are that
it has no name of its own (`<lambda>` is what shows up in tracebacks) and
that its body must be a single expression.

No `return`, because there is nothing to return *from* — the expression is
the result. Try to put a statement in one (`lambda x: print(x); x + 1`) and
it is a `SyntaxError`.

`double = lambda x: x * 2` is legal and is the one form to avoid. If you
are giving it a name anyway, use `def` — you get a real name in the
traceback and somewhere to put a docstring, for the same number of
characters.

In [ ]:
def value_double(x):
    return x * 2

double = lambda x: x * 2

print(value_double(5))
print(double(5))
print(type(double))
print(double)

### Question 2

Calling one without naming it. -> `10`, `42`, `no arguments at all`.

The parentheses around the lambda are required: `lambda x: x * 2(5)` would
parse as `lambda x: x * (2(5))` and raise, because `:` swallows as much as
it can to its right.

This immediately-invoked form is a curiosity, not a technique — it defines
a function and throws it away in the same breath. Slide 62 uses it to show
the equivalence with `def`, which is the only good reason to write one.

The third shows a lambda can take no arguments at all. Still needs the
`lambda` keyword, the colon, and `()` to call it.

In [ ]:
print((lambda x: x * 2)(5))
print((lambda x, y: x * y)(6, 7))
print((lambda: "no arguments at all")())

### Question 3

All four argument styles. -> `6`, `6`, `6`, `6`.

Everything worksheet 10 covered works on a lambda: positional, defaults,
`*args`, `**kwargs`. There is no restriction on the *parameters* — only on
the body, which must be one expression.

All four give 6, which is the point of the slide: these are not four
different kinds of function, they are four ways of getting values into the
same kind of function.

And a lambda with `**kwargs` in it is a strong hint you have outgrown
lambdas. Use `def`.

In [ ]:
print((lambda x, y, z: x + y + z)(1, 2, 3))
print((lambda x, y, z=3: x + y + z)(1, 2))
print((lambda *args: sum(args))(1, 2, 3))
print((lambda **kwargs: sum(kwargs.values()))(one=1, two=2, three=3))

PART B — filter and map

### Question 4

`filter`. -> `[22, 4, 8]`, then `<filter object at 0x…>`.

`filter` keeps the items for which the function returns something truthy.
Seven in, three out — like a comprehension's trailing `if`.

The second line is why `list(...)` was needed for the first. **`filter`
returns a lazy iterator, not a list.** Nothing has been computed yet at the
point it is printed; the lambda has not run once. It runs as something
pulls items out.

That is usually a good thing — a filter over a million rows costs nothing
until you consume it — and it is the setup for Q10, which is where it bites.

In [ ]:
even_numbers = list(filter(lambda x: x % 2 == 0, numbers))
print(even_numbers)

print(filter(lambda x: x % 2 == 0, numbers))

### Question 5

`map`. -> `[1, 0, 1, 0, 1, 1, 0] 7 from 7`, then `['EXTRACT', 'LOAD', 'TRANSFORM', 'QA'] 4 from 4`.

`map` applies the function to **every** item, so the output is always the
same length as the input. `filter` decides what to keep and can shorten it.

That is exactly worksheet 03 Q5 — the map form and the filter form of a
comprehension — under different names. `map`/`filter` are the older,
function-shaped way of writing the same two ideas.

Both are lazy, both needed `list()`.

In [ ]:
remainder = list(map(lambda x: x % 2, numbers))
print(remainder, len(remainder), "from", len(numbers))

shouted = list(map(lambda w: w.upper(), words))
print(shouted, len(shouted), "from", len(words))

### Question 6

Three ways to the same answer. -> `[484, 16, 64]` three times, then `True`.

Identical results. Now compare them as things to read:

- `list(map(lambda x: x ** 2, filter(lambda x: x % 2 == 0, numbers)))` —
  reads inside-out, mentions `numbers` in the middle, and needs two
  lambdas and a `list()`.
- `[x ** 2 for x in numbers if x % 2 == 0]` — reads left to right, once.
- the loop — three lines, and the one you write when the body grows.

In Python the comprehension is the idiomatic choice; `map`/`filter` are
older and the language's own style guidance prefers comprehensions for
exactly this case. Know `map` and `filter` because you will read them, and
because `map(str.upper, words)` with no lambda is genuinely neat — but do
not reach for them first.

In [ ]:
a = list(map(lambda x: x ** 2, filter(lambda x: x % 2 == 0, numbers)))

b = [x ** 2 for x in numbers if x % 2 == 0]

c = []
for x in numbers:
    if x % 2 == 0:
        c.append(x ** 2)

print(a)
print(b)
print(c)
print(a == b == c)

PART C — Where lambdas actually earn their keep

### Question 7

`sorted(key=...)`. -> `['bo', 'dee', 'ana', 'cai']` by years; `['cai', 'ana', 'dee', 'bo']` descending; `['ana', 'bo', 'cai', 'dee']` by name.

**This is what lambdas are for.** `key=` takes a function, calls it once per
item, and sorts on whatever comes back — so a one-expression throwaway
function is exactly the right shape, and naming it would be noise.

`sorted` returns a new list and leaves `staff` alone, which is worth
checking after every sort: `.sort()` would rearrange it in place and return
`None`.

The key function runs once per element, not once per comparison, so an
expensive key is fine. And ties keep their original order — Python's sort
is stable — which is what lets you sort by one field and then another to
get a two-level ordering.

In [ ]:
by_years = sorted(staff, key=lambda person: person["years"])
print([p["name"] for p in by_years])

by_years_desc = sorted(staff, key=lambda person: person["years"], reverse=True)
print([p["name"] for p in by_years_desc])

by_name = sorted(staff, key=lambda person: person["name"])
print([p["name"] for p in by_name])

### Question 8

`max` and `min` with a key. -> `cai 9`, `bo 2`, then `{'name': 'ana', 'role': 'engineer', 'years': 6}`.

`max(staff, key=...)` returns **the item**, not the key value — `cai`'s whole
record, not the number 9. That is usually what you want and is easy to get
backwards: `max(p["years"] for p in staff)` gives you 9 and no idea whose.

The last line is valid code answering a meaningless question. `max` over a
**text** field returns whatever sorts last alphabetically — `engineer`
beats `analyst` — and among the two engineers it returns the first one
found, so the answer depends on list order. Nothing raises. It is worksheet
06 Q10's lesson in a single expression: the computer will happily rank
things that have no ranking.

In [ ]:
most = max(staff, key=lambda p: p["years"])
least = min(staff, key=lambda p: p["years"])
print(most["name"], most["years"])
print(least["name"], least["years"])

print(max(staff, key=lambda p: p["role"]))
# It means "the record whose role sorts last alphabetically" -- engineer vs
# analyst, so an engineer, and which engineer depends on list order. The code
# is valid and the question is nonsense: max() over a text field answers
# nothing anybody wanted to ask.

### Question 9

No lambda needed. -> `['QA', 'Load', 'extract', 'transform']` by length; `['EXTRACT', 'LOAD', 'TRANSFORM', 'QA']`; `['extract', 'Load', 'QA', 'transform']` case-insensitively; and `['Load', 'QA', 'extract', 'transform']` plain.

`key=len` and `key=lambda w: len(w)` do the same thing, and the first says
it in four characters. **If your lambda just calls one function on its
argument, pass that function.**

`str.upper` works as a plain function because `words[0].upper()` is really
`str.upper(words[0])` — the method looked up on the class, taking the
instance as its argument. So `map(str.upper, words)` needs no lambda at
all.

The last two lines are the useful contrast. Plain `sorted(words)` puts
`Load` and `QA` first, because every capital letter sorts before every
lowercase one — extra practice 01 Q6. `key=str.lower` is how you sort text
the way a human would read it.

In [ ]:
print(sorted(words, key=len))
print(list(map(str.upper, words)))
print(sorted(words, key=str.lower))
print(sorted(words))          # for comparison -- capitals sort first

PART D — Lazy and one-shot

### Question 10

One-shot iterators. -> `[2, 44, 70, 8, 10, 14, 16]`, then **`[]`**, then `164`.

Same object, two `list()` calls, and the second one is empty. No error, no
warning.

A `map` object is an **iterator**: it produces values once, on demand, and
then it is exhausted. The first `list()` consumed all seven; the second
found nothing left. `filter`, `zip`, `enumerate` and generator expressions
all behave this way.

This is the bug that costs an afternoon, because it usually appears as "the
second loop over my results does nothing" or "my count is zero but the
print above shows rows". The output is empty rather than wrong, which
makes it look like a data problem rather than a code problem.

Two fixes: wrap it in `list(...)` **once** and keep that list, or rebuild
the map each time you need it (as the `sum` line does). And note
`len(map(...))` raises — an iterator does not know how many items it will
produce until it has produced them.

In [ ]:
doubled = map(lambda x: x * 2, numbers)

print(list(doubled))
print(list(doubled))          # same object, second time

print(sum(map(lambda x: x * 2, numbers)))

# len(map(...)) would raise TypeError: object of type 'map' has no len().
# A map object does not know how many items it will produce until it has
# produced them.

### Question 11

A key that is not there. -> the working sort prints `['bo', 'dee', 'ana', 'cai']`, then `KeyError: 'salary'`.

The key function is called once per item and the very first one has no
`salary`, so the sort fails before comparing anything.

Good. The alternative — `key=lambda p: p.get("salary", 0)` — does not
raise, and that is precisely the problem: it silently decides that a
missing salary is zero, sorts those people to the bottom, and produces a
ranking that looks complete. If four of your fifty records lack the field,
nothing in the output says so.

If you use a default in a key function, **count the defaults you applied
and print the count**. That is worksheet 07 Q11 again: a result is not a
result without the number of rows it quietly assumed something about.

In [ ]:
print([p["name"] for p in sorted(staff, key=lambda p: p["years"])])

# This is SUPPOSED to raise: KeyError: 'salary'. The key function runs once
# per item, and the very first one has no such field.
#
# For a field only some records have, use .get() with a default that sorts
# where you want the gaps to land:
#     sorted(staff, key=lambda p: p.get("salary", 0))
# -- and be aware that you have just decided missing means zero, which will
# put those people at the bottom and nobody will be told.
print([p["name"] for p in sorted(staff, key=lambda p: p["salary"])])